In [0]:
 #===============================================================
# CELL 1 — CONFIG & IMPORTS
# ================================================================
import time
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark.ml.fpm import FPGrowth

TABLE_NAME   = "workspace.default.final_table_clean"
ORDER_COL    = "order_id"
PRODUCT_COL  = "item_name"
QUANTITY_COL = "quantity"
OUTPUT_TABLE = "workspace.default.mba_rules"
OUTPUT_PATH  = "/tmp/mba_outputs/"

spark.conf.set("spark.sql.shuffle.partitions", "200")

print("Config loaded successfully")
print(f"  Table   : {TABLE_NAME}")
print(f"  Order   : {ORDER_COL}")
print(f"  Product : {PRODUCT_COL}")
print(f"  Qty     : {QUANTITY_COL}")

Config loaded successfully
  Table   : workspace.default.final_table_clean
  Order   : order_id
  Product : item_name
  Qty     : quantity


In [0]:
# ================================================================
# CELL 2 — DATA LOADING & VALIDATION
# ================================================================
df = spark.table(TABLE_NAME)
 
df.printSchema()
display(df.limit(5))
 
total_rows     = df.count()
total_orders   = df.select(ORDER_COL).distinct().count()
total_products = df.select(PRODUCT_COL).distinct().count()
 
missing_cols = [c for c in [ORDER_COL, PRODUCT_COL, QUANTITY_COL] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")
 
print(f"Rows      : {total_rows:,}")
print(f"Orders    : {total_orders:,}")
print(f"Products  : {total_products:,}")
print("Validation passed")
 

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- branch: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_type: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- rating: double (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- source: string (nullable = true)



order_id,order_date,hour,category,item_name,price,quantity,discount,total_amount,branch,payment_method,order_type,customer_id,rating,is_weekend,source
1,2021-09-25,18,مشروبات,عصير مانجو,34.83,3,0.0,104.5,القاهرة,Cash,Dine-in,2499,4.0,true,JSON
2,2024-06-22,14,طواجن,طاجن بامية,120.27,3,0.0,360.81,القاهرة,Cash,Takeaway,80347,4.0,true,JSON
3,2024-03-26,23,مشروبات,شاي,24.87,2,0.0,49.75,طنطا,Wallet,Dine-in,78982,3.0,false,JSON
4,2023-11-07,19,محاشي,محشي باذنجان,109.01,3,0.0,327.02,القاهرة,Cash,Delivery,101142,3.0,false,JSON
5,2023-02-08,22,مشروبات,عصير مانجو,13.93,2,0.0,27.85,الإسكندرية,Cash,Dine-in,4076,4.0,false,JSON


Rows      : 11,110,000
Orders    : 2,500,000
Products  : 15
Validation passed


In [0]:

# ================================================================
# CELL 3 — DATA CLEANING
# ================================================================
rows_before = df.count()
 
df_clean = df.dropna(subset=[ORDER_COL, PRODUCT_COL, QUANTITY_COL])
 
df_clean = df_clean.filter(F.col(QUANTITY_COL) > 0)
 
df_clean = df_clean.withColumn(ORDER_COL, F.col(ORDER_COL).cast(T.StringType()))
 
df_clean = df_clean.withColumn(QUANTITY_COL, F.col(QUANTITY_COL).cast(T.IntegerType()))
 
df_clean = df_clean.withColumn(PRODUCT_COL, F.trim(F.col(PRODUCT_COL)))
 
df_clean = df_clean.groupBy(ORDER_COL, PRODUCT_COL).agg(
    F.sum(QUANTITY_COL).alias(QUANTITY_COL)
)
 
rows_after = df_clean.count()
 
print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows after  cleaning : {rows_after:,}")
print(f"Rows removed         : {rows_before - rows_after:,}")
print("Cleaning done")
 

Rows before cleaning : 11,110,000
Rows after  cleaning : 9,035,935
Rows removed         : 2,074,065
Cleaning done


In [0]:

# ================================================================
# CELL 4 — BUILD TRANSACTION BASKETS
# ================================================================
df_baskets = df_clean.groupBy(ORDER_COL).agg(
    F.collect_list(PRODUCT_COL).alias("items")
)
 
df_baskets = df_baskets.filter(F.size(F.col("items")) >= 2)
 
total_baskets   = df_baskets.count()
avg_basket_size = df_baskets.select(F.avg(F.size("items"))).collect()[0][0]
max_basket_size = df_baskets.select(F.max(F.size("items"))).collect()[0][0]
 
print(f"Total Baskets    : {total_baskets:,}")
print(f"Avg Basket Size  : {avg_basket_size:.2f}")
print(f"Max Basket Size  : {max_basket_size}")
print("Baskets built")
 
display(df_baskets.limit(5))
 

Total Baskets    : 1,957,993
Avg Basket Size  : 4.34
Max Basket Size  : 9
Baskets built


order_id,items
296,"List(عصير قصب, عصير مانجو, محشي كوسة, كباب, طحينة, كفتة)"
467,"List(محشي كوسة, عصير قصب, كفتة, طاجن فراخ, عصير مانجو, كباب, شاي)"
675,"List(شيش طاووق, طاجن فراخ, بابا غنوج, عصير مانجو, محشي كوسة)"
691,"List(عصير قصب, طاجن فراخ, طاجن بامية, طحينة, عصير مانجو)"
829,"List(محشي باذنجان, طاجن فراخ, كفتة, طحينة, بابا غنوج)"


In [0]:

 
 
# ================================================================
# CELL 5 — FP-GROWTH MODEL
# ================================================================
min_support = 0.01 if total_baskets > 1000 else 0.005
 
fp = FPGrowth(
    itemsCol="items",
    minSupport=min_support,
    minConfidence=0.10,
    numPartitions=200
)
 
start_time = time.time()
model      = fp.fit(df_baskets)
elapsed    = time.time() - start_time
 
freq_itemsets     = model.freqItemsets
association_rules = model.associationRules
 
n_freq  = freq_itemsets.count()
n_rules = association_rules.count()
 
print(f"Model trained in  : {elapsed:.1f} seconds")
print(f"Frequent itemsets : {n_freq:,}")
print(f"Association rules : {n_rules:,}")
 
if n_rules == 0:
    print("WARNING: No rules found. Try lowering minSupport to 0.005 or minConfidence to 0.05")
else:
    display(association_rules.limit(10))
 
 

Model trained in  : 17.9 seconds
Frequent itemsets : 1,173
Association rules : 4,056


antecedent,consequent,confidence,lift,support
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(طحينة),0.4869000117357118,1.200962454512227,0.03390308341245347
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(كباب),0.48430348550639596,1.1960767881589998,0.03372228603473046
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(محشي ورق عنب),0.4109626217580096,1.1996951593414718,0.028615526204639138
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(كفتة),0.3229961272151156,1.185100527530059,0.022490376625452696
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(طاجن ملوخية),0.3250058678558855,1.1942759926896556,0.022630315838718523
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(عصير قصب),0.32374427884051166,1.189986280397401,0.022542470785135594
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(بابا غنوج),0.32804981809646755,1.206357928958215,0.022842267566840126
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(طاجن بامية),0.3238763055979345,1.1911804391815013,0.02255166387213846
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(محشي باذنجان),0.22772415209482455,1.1820124852742324,0.015856542898774407
"List(طاجن فراخ, محشي كوسة, عصير مانجو)",List(شيش طاووق),0.225743750733482,1.1757132240531953,0.01571864659373144


In [0]:
# CELL 6 FIXED — Strength thresholds adjusted for restaurant data
# Actual lift range in your data: 1.08 → 1.10

df_rules = association_rules.withColumn(
    "Product_A", F.array_join(F.col("antecedent"), " | ")
)
df_rules = df_rules.withColumn(
    "Product_B", F.array_join(F.col("consequent"), " | ")
)
df_rules = df_rules.withColumn("support",    F.round("support",    6))
df_rules = df_rules.withColumn("confidence", F.round("confidence", 4))
df_rules = df_rules.withColumn("lift",       F.round("lift",       4))
df_rules = df_rules.withColumn("ant_len",    F.size("antecedent"))
df_rules = df_rules.withColumn("con_len",    F.size("consequent"))
df_rules = df_rules.withColumn(
    "Score",
    F.round((F.col("confidence") * 0.6) + (F.col("lift") * 0.4), 4)
)

# ✅ Thresholds adjusted to match actual lift range (1.08 - 1.10)
df_rules = df_rules.withColumn(
    "Strength",
    F.when(F.col("lift") >= 1.10, "Very Strong")
     .when(F.col("lift") >= 1.08, "Strong")
     .when(F.col("lift") >= 1.05, "Medium")
     .otherwise("Weak")
)

display(df_rules.select("Product_A","Product_B","lift","confidence","Strength","Score").orderBy(F.col("lift").desc()).limit(10))

print("Strength Distribution:")
display(df_rules.groupBy("Strength").count().orderBy(F.col("count").desc()))

Product_A,Product_B,lift,confidence,Strength,Score
كباب | طحينة | عصير مانجو,بابا غنوج,1.2102,0.3291,Very Strong,0.6815
محشي كوسة | طحينة | عصير مانجو,بابا غنوج,1.209,0.3288,Very Strong,0.6809
محشي ورق عنب | طحينة | عصير مانجو,عصير قصب,1.2078,0.3286,Very Strong,0.6803
كباب | طحينة | عصير مانجو,طاجن بامية,1.2076,0.3283,Very Strong,0.68
محشي كوسة | كباب | عصير مانجو,شاي,1.2076,0.1229,Very Strong,0.5568
كباب | طحينة | عصير مانجو,محشي كوسة,1.2074,0.4136,Very Strong,0.7311
كباب | طحينة | عصير مانجو,طاجن ملوخية,1.2068,0.3284,Very Strong,0.6798
طاجن فراخ | طحينة | عصير مانجو,بابا غنوج,1.2064,0.3281,Very Strong,0.6794
طاجن فراخ | محشي كوسة | عصير مانجو,بابا غنوج,1.2064,0.328,Very Strong,0.6794
كباب | طحينة | عصير مانجو,محشي ورق عنب,1.2062,0.4132,Very Strong,0.7304


Strength Distribution:


Strength,count
Very Strong,3899
Strong,157


In [0]:
 
# ================================================================
# CELL 7 — FILTERING & RANKING
# ================================================================
window_rank = Window.orderBy(F.col("Score").desc())
 
df_final = df_rules.filter(
    (F.col("ant_len") == 1) & (F.col("con_len") == 1)
)
 
df_final = df_final.filter(F.col("lift") >= 1.0)
 
df_final = df_final.withColumn("rank", F.row_number().over(window_rank))
 
df_final = df_final.select(
    "rank", "Product_A", "Product_B",
    "support", "confidence", "lift", "Strength", "Score"
)
 
df_final = df_final.orderBy("rank").limit(500)
 
n_final = df_final.count()
print(f"Rules after filter : {n_final:,}")
display(df_final)
 

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Rules after filter : 210


rank,Product_A,Product_B,support,confidence,lift,Strength,Score
1,شاي,عصير مانجو,0.051454,0.5057,1.096,Strong,0.7418
2,سلطة بلدي,عصير مانجو,0.096854,0.5049,1.0945,Strong,0.7407
3,محشي باذنجان,عصير مانجو,0.097184,0.5044,1.0934,Strong,0.74
4,بابا غنوج,عصير مانجو,0.137035,0.5039,1.0923,Strong,0.7393
5,طاجن بامية,عصير مانجو,0.136819,0.5032,1.0907,Strong,0.7382
6,عصير قصب,عصير مانجو,0.13685,0.503,1.0903,Strong,0.7379
7,طاجن ملوخية,عصير مانجو,0.136863,0.5029,1.0901,Strong,0.7378
8,كفتة,عصير مانجو,0.137054,0.5029,1.09,Strong,0.7377
9,شيش طاووق,عصير مانجو,0.096557,0.5029,1.09,Strong,0.7377
10,محشي ورق عنب,عصير مانجو,0.171612,0.501,1.0859,Strong,0.735


In [0]:

# ================================================================
# CELL 8 — SAVE OUTPUTS
# ================================================================
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE)

print(f"Rules saved      : {OUTPUT_TABLE}")

window_freq = Window.orderBy(F.col("freq").desc())

df_freq = freq_itemsets.filter(F.size("items") == 1)
df_freq = df_freq.withColumn("product", F.col("items")[0])
df_freq = df_freq.withColumn("frequency_rank", F.row_number().over(window_freq))
df_freq = df_freq.select("product", "freq", "frequency_rank").orderBy("frequency_rank")

df_freq.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE + "_itemsets")

print(f"Itemsets saved   : {OUTPUT_TABLE}_itemsets")
print("All outputs saved to Delta tables successfully")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Rules saved      : workspace.default.mba_rules
Itemsets saved   : workspace.default.mba_rules_itemsets
All outputs saved to Delta tables successfully


In [0]:
 
# ================================================================
# CELL 9 — SUMMARY REPORT
# ================================================================
strength_dist = df_final.groupBy("Strength").count().orderBy(F.col("count").desc()).collect()
top5          = df_final.limit(5).collect()
top_products  = df_freq.limit(10).collect()
 
print("=" * 52)
print("   Market Basket Analysis — Final Summary")
print("=" * 52)
print(f"  Orders Analyzed    : {total_orders:,}")
print(f"  Unique Products    : {total_products:,}")
print(f"  Frequent Itemsets  : {n_freq:,}")
print(f"  Rules Generated    : {n_rules:,}")
print(f"  Rules After Filter : {n_final:,}")
print(f"  Training Time      : {elapsed:.1f} sec")
print()
print("  Strength Distribution:")
for row in strength_dist:
    print(f"    {row['Strength']:<12} : {row['count']:>4} rules")
print()
print("  Top 5 Rules:")
for row in top5:
    print(f"    #{row['rank']}  {row['Product_A']}  ->  {row['Product_B']}")
    print(f"       Lift={row['lift']}  Conf={row['confidence']}  [{row['Strength']}]")
print()
print("  Top 10 Most Frequent Items:")
for row in top_products:
    print(f"    #{row['frequency_rank']:>2}  {row['product']:<30} freq={row['freq']}")
print("=" * 52)
print("\nMBA Analysis completed successfully")
 

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


   Market Basket Analysis — Final Summary
  Orders Analyzed    : 2,500,000
  Unique Products    : 15
  Frequent Itemsets  : 1,173
  Rules Generated    : 4,056
  Rules After Filter : 210
  Training Time      : 17.9 sec

  Strength Distribution:
    Strong       :  142 rules
    Very Strong  :   68 rules

  Top 5 Rules:
    #1  شاي  ->  عصير مانجو
       Lift=1.096  Conf=0.5057  [Strong]
    #2  سلطة بلدي  ->  عصير مانجو
       Lift=1.0945  Conf=0.5049  [Strong]
    #3  محشي باذنجان  ->  عصير مانجو
       Lift=1.0934  Conf=0.5044  [Strong]
    #4  بابا غنوج  ->  عصير مانجو
       Lift=1.0923  Conf=0.5039  [Strong]
    #5  طاجن بامية  ->  عصير مانجو
       Lift=1.0907  Conf=0.5032  [Strong]

  Top 10 Most Frequent Items:
    # 1  عصير مانجو                     freq=903320
    # 2  طحينة                          freq=793819
    # 3  كباب                           freq=792811
    # 4  محشي ورق عنب                   freq=670722
    # 5  محشي كوسة                      freq=670701
    # 6  طاج